# 06 — Correspondences Between Shapes

**The question:** what does it mean for a point on one shape to "correspond" to a point on another, and why is finding such correspondences useful but difficult?

## Intuition

Imagine two cats in different poses. The tip of the nose on the first cat should correspond to the tip of the nose on the second. A **correspondence** is a systematic assignment of such pairs — a map that tells us, for every point on shape A, which point on shape B plays the same anatomical/semantic role.

Correspondences are fundamental in:
- **Shape analysis**: compare two shapes by comparing corresponding functions
- **Animation**: deform shape A into shape B point by point
- **Segmentation transfer**: propagate a label map from a labelled shape to a new one
- **Statistical shape models**: build an atlas of shapes over a population

## Minimal math

A **point-to-point (p2p) map** from shape $A$ to shape $B$ is a function:

$$T: V_A \to V_B, \qquad T \text{ stored as an array of length } n_B$$

where $T[j] = i$ means vertex $j$ of $B$ corresponds to vertex $i$ of $A$.

The **pullback** of a function $f: A \to \mathbb{R}$ via $T$ is:

$$(T^* f)(j) = f(T[j])$$

This is how we *transfer* a function defined on $A$ onto $B$: just index into $f$ with the map array.

**Geodesic error** measures correspondence quality:

$$\varepsilon = \frac{1}{n_B} \sum_{j=1}^{n_B} \frac{d_B(T[j],\, T^{\mathrm{gt}}[j])}{\mathrm{diam}(B)}$$

where $d_B$ is geodesic distance and $\mathrm{diam}(B)$ is the shape diameter. A perfect map has $\varepsilon = 0$; a random map has $\varepsilon \approx 0.25$ on typical benchmarks.

In [1]:
import numpy as np

from geomfum.dataset import NotebooksDataset
from geomfum.plot import MeshPlotter
from geomfum.shape import TriangleMesh

In [2]:
dataset = NotebooksDataset()
mesh_a = TriangleMesh.from_file(dataset.get_filename("cat-00"))
mesh_b = TriangleMesh.from_file(dataset.get_filename("lion-00"))

print(f"Shape A (cat):  {mesh_a.n_vertices} vertices")
print(f"Shape B (lion): {mesh_b.n_vertices} vertices")

Shape A (cat):  7207 vertices
Shape B (lion): 5000 vertices


## A trivial p2p map: nearest neighbour in 3D

The simplest map assigns to each vertex of $B$ the nearest vertex of $A$ in Euclidean distance. This ignores the intrinsic geometry entirely and produces a poor correspondence — but it shows the data structure.

In [13]:
from geomfum.convert import NeighborFinder

verts_a = mesh_a.vertices  # (n_a, 3)
verts_b = mesh_b.vertices  # (n_b, 3)


# Build a simple nearest-neighbour p2p map B -> A
nn = NeighborFinder()
p2p_nn = nn(mesh_b.vertices, mesh_a.vertices)  # (n_b,)


print(f"p2p array shape: {p2p_nn.shape}  (one index into A for each vertex of B)")
print(f"Maps vertex 0 of B to vertex {p2p_nn[0]} of A")

p2p array shape: (5000, 1)  (one index into A for each vertex of B)
Maps vertex 0 of B to vertex [6150] of A


## Transfer a function via p2p

Given any function $f$ on $A$ and a map $T$, the pullback $(T^* f)(j) = f[T[j]]$ transfers $f$ to $B$ with a single index operation.

In [14]:
# Function on A: height (z-coordinate)
f_a = verts_a[:, 2]  # shape (n_a,)

# Pullback to B
f_b_transferred = f_a[p2p_nn]  # (n_b,)

print(f"Transferred function shape: {f_b_transferred.shape}")

Transferred function shape: (5000, 1)


In [9]:
# Show original function on A (cat)
plotter = MeshPlotter.from_registry(which="polyscope")
plotter.add_mesh(mesh_a)
plotter.set_vertex_scalars(f_a)
print("Source function on cat:")
plotter.show()

Source function on cat:


In [10]:
# Show transferred function on B (lion) via nearest-neighbour map
plotter = MeshPlotter.from_registry(which="polyscope")
plotter.add_mesh(mesh_b)
plotter.set_vertex_scalars(f_b_transferred)
print("Transferred to lion (nearest-neighbour — not semantically meaningful):")
plotter.show()

Transferred to lion (nearest-neighbour — not semantically meaningful):


## Why finding correspondences is hard

A good correspondence must be:
- **Semantically consistent**: nose maps to nose, paw to paw
- **Smooth**: nearby vertices should map to nearby vertices
- **Robust to pose/scale changes**: the same map should work regardless of the shape's orientation or size

Challenges:

1. **Symmetry ambiguity** — the left paw and right paw look identical locally; any algorithm must break this symmetry
2. **Different discretisations** — $n_A \neq n_B$; we cannot rely on vertex indices matching
3. **Non-isometric deformations** — cat vs lion is not isometric (different body proportions)
4. **No ground truth** — for real data the correct correspondence is unknown

In [11]:
# Illustration: the identity map on cat-00 is a "perfect" self-correspondence
p2p_identity = np.arange(mesh_a.n_vertices)  # vertex i -> vertex i

f_self = f_a[p2p_identity]  # trivially the same function
print(f"Self-map error: {np.abs(f_self - f_a).max():.2e}  (should be 0)")

Self-map error: 0.00e+00  (should be 0)


## Where to go next

- [07 — Functional Maps](./07_functional_maps.ipynb): a smarter representation that avoids the $n \times n$ cost
- [How to compute a pointwise map from a functional map?](../how_to/10_pointwise_from_functional.ipynb)